# AVL Tree (Self-Balancing BST) for City Storage

Extends the BST by maintaining a balance factor at every node and performing
rotations (LL, RR, LR, RL) after insertions/deletions to keep the tree height
within O(log n) at all times — unlike a plain BST, which can degrade to O(n)
in the worst case.

- Guaranteed: O(log n) insert / search / delete, even in worst case
- Trade-off: extra bookkeeping (height tracking + rotations) on every insert/delete

In [1]:
class CityData:
    """Container for the data associated with a city node."""
    def __init__(self, name, latitude, longitude, population, distance=0.0):
        self.name = name
        self.latitude = latitude
        self.longitude = longitude
        self.population = population
        self.distance = distance

    def __repr__(self):
        return f"City({self.name}, pop={self.population}, dist={self.distance})"

In [2]:
class AVLNode:
    def __init__(self, city: CityData):
        self.city = city
        self.left = None
        self.right = None
        self.height = 1  # height of this node's subtree (leaf = 1)


class AVLTree:
    """
    Self-balancing BST keyed on city name.
    Guarantees O(log n) height via rotations after insert/delete.
    """

    def __init__(self):
        self.root = None
        self.size = 0

    # ---------- helpers ----------
    def _height(self, node):
        return node.height if node else 0

    def _balance_factor(self, node):
        return self._height(node.left) - self._height(node.right) if node else 0

    def _update_height(self, node):
        node.height = 1 + max(self._height(node.left), self._height(node.right))

    def _rotate_right(self, y):
        x = y.left
        T2 = x.right
        x.right = y
        y.left = T2
        self._update_height(y)
        self._update_height(x)
        return x

    def _rotate_left(self, x):
        y = x.right
        T2 = y.left
        y.left = x
        x.right = T2
        self._update_height(x)
        self._update_height(y)
        return y

    def _rebalance(self, node):
        self._update_height(node)
        balance = self._balance_factor(node)

        # Left heavy
        if balance > 1:
            if self._balance_factor(node.left) < 0:
                node.left = self._rotate_left(node.left)  # LR case
            return self._rotate_right(node)  # LL case

        # Right heavy
        if balance < -1:
            if self._balance_factor(node.right) > 0:
                node.right = self._rotate_right(node.right)  # RL case
            return self._rotate_left(node)  # RR case

        return node

    # ---------- insert ----------
    def insert(self, city: CityData):
        self.root = self._insert(self.root, city)
        self.size += 1

    def _insert(self, node, city):
        if node is None:
            return AVLNode(city)
        if city.name < node.city.name:
            node.left = self._insert(node.left, city)
        elif city.name > node.city.name:
            node.right = self._insert(node.right, city)
        else:
            node.city = city
            self.size -= 1
            return node
        return self._rebalance(node)

    # ---------- search ----------
    def search(self, name):
        return self._search(self.root, name)

    def _search(self, node, name):
        if node is None:
            return None
        if name == node.city.name:
            return node.city
        elif name < node.city.name:
            return self._search(node.left, name)
        else:
            return self._search(node.right, name)

    # ---------- delete ----------
    def delete(self, name):
        self.root, deleted = self._delete(self.root, name)
        if deleted:
            self.size -= 1
        return deleted

    def _delete(self, node, name):
        if node is None:
            return node, False

        if name < node.city.name:
            node.left, deleted = self._delete(node.left, name)
        elif name > node.city.name:
            node.right, deleted = self._delete(node.right, name)
        else:
            deleted = True
            if node.left is None and node.right is None:
                return None, deleted
            if node.left is None:
                return node.right, deleted
            if node.right is None:
                return node.left, deleted
            successor = self._min_node(node.right)
            node.city = successor.city
            node.right, _ = self._delete(node.right, successor.city.name)

        return self._rebalance(node), deleted

    def _min_node(self, node):
        while node.left is not None:
            node = node.left
        return node

    # ---------- utilities ----------
    def inorder(self):
        result = []
        self._inorder(self.root, result)
        return result

    def _inorder(self, node, result):
        if node:
            self._inorder(node.left, result)
            result.append(node.city)
            self._inorder(node.right, result)

    def height(self):
        return self._height(self.root) - 1 if self.root else -1

## Sanity Test — including worst-case ordered insertion

In [3]:
avl = AVLTree()
cities = ["Kathmandu", "Pokhara", "Biratnagar", "Lalitpur", "Bharatpur", "Butwal", "Dharan"]
for i, name in enumerate(cities):
    avl.insert(CityData(name, 27.0 + i, 85.0 + i, 100000 * (i + 1)))

print("Inorder:", avl.inorder())
print("Height:", avl.height())
print("Search Butwal:", avl.search("Butwal"))

avl.delete("Pokhara")
print("After delete:", avl.inorder())
print("Height after delete:", avl.height())

Inorder: [City(Bharatpur, pop=500000, dist=0.0), City(Biratnagar, pop=300000, dist=0.0), City(Butwal, pop=600000, dist=0.0), City(Dharan, pop=700000, dist=0.0), City(Kathmandu, pop=100000, dist=0.0), City(Lalitpur, pop=400000, dist=0.0), City(Pokhara, pop=200000, dist=0.0)]
Height: 3
Search Butwal: City(Butwal, pop=600000, dist=0.0)
After delete: [City(Bharatpur, pop=500000, dist=0.0), City(Biratnagar, pop=300000, dist=0.0), City(Butwal, pop=600000, dist=0.0), City(Dharan, pop=700000, dist=0.0), City(Kathmandu, pop=100000, dist=0.0), City(Lalitpur, pop=400000, dist=0.0)]
Height after delete: 2


## Key demonstration: AVL stays balanced even with sorted (worst-case) input

In [4]:
# Insert already-sorted city names — this is the case that makes a plain BST
# degrade into a linked list (height = n). AVL should stay ~log(n).
avl_sorted = AVLTree()
sorted_names = [f"City{str(i).zfill(3)}" for i in range(1, 16)]  # City001..City015
for i, name in enumerate(sorted_names):
    avl_sorted.insert(CityData(name, 0, 0, 0))

print(f"Inserted {len(sorted_names)} cities in sorted order")
print("AVL height:", avl_sorted.height())
print("Expected plain-BST height for sorted input:", len(sorted_names) - 1)

Inserted 15 cities in sorted order
AVL height: 3
Expected plain-BST height for sorted input: 14
